In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer
import joblib

# Load the dataset
df = pd.read_csv('election_prediction_data.csv')

# Convert relevant columns to appropriate types
df['is_winner'] = df['is_winner'].astype(int)
df['candidate_is_incumbent'] = df['candidate_is_incumbent'].astype(int)

# Split into features and target
X = df.drop(['is_winner', 'election_id', 'position_id', 'candidate_id'], axis=1)
y = df['is_winner']

# Identify feature types
numeric_features = [
    'election_year', 'election_month', 'election_duration_hours',
    'max_candidates', 'candidate_previous_wins', 'candidate_previous_losses',
    'position_avg_voters', 'position_avg_turnout', 'votes_received',
    'vote_percentage', 'institution_voter_turnout_avg',
    'institution_votes_for_own_candidates_pct', 'votes_first_hour_pct',
    'votes_last_hour_pct', 'votes_peak_hour_pct', 'closest_competitor_vote_diff',
    'second_competitor_vote_diff', 'fraud_risk_avg', 'fraud_indicators_count'
]

categorical_features = [
    'election_name', 'election_day_of_week', 'academic_year',
    'institution_level', 'institution_name', 'position_name',
    'position_institution', 'candidate_institution',
    'candidate_academic_year', 'position_winner_institution_last_3',
    'position_winner_academic_year_last_3'
]

# Create preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [2]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define models
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        max_depth=10
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    )
}

# Train and evaluate models
best_model = None
best_score = 0

for name, model in models.items():
    # Create complete pipeline
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train model
    clf.fit(X_train, y_train)
    
    # Evaluate
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy:.3f}")
    print(classification_report(y_test, y_pred))
    
    # Track best model
    if accuracy > best_score:
        best_score = accuracy
        best_model = clf

# Save the best model
joblib.dump(best_model, 'election_winner_predictor.joblib')

/home/tarxemo/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


Random Forest Accuracy: 0.994
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       276
           1       1.00      0.97      0.99        69

    accuracy                           0.99       345
   macro avg       1.00      0.99      0.99       345
weighted avg       0.99      0.99      0.99       345



/home/tarxemo/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


Gradient Boosting Accuracy: 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       276
           1       1.00      1.00      1.00        69

    accuracy                           1.00       345
   macro avg       1.00      1.00      1.00       345
weighted avg       1.00      1.00      1.00       345



['election_winner_predictor.joblib']

In [3]:
def predict_current_election(current_election_data_path, model_path='election_winner_predictor.joblib'):
    # Load current election data
    current_df = pd.read_csv(current_election_data_path)
    
    # Load trained model
    model = joblib.load(model_path)
    
    # Prepare features (same as training)
    X_current = current_df.drop(['election_id', 'position_id', 'candidate_id'], axis=1, errors='ignore')
    
    # Make predictions
    predictions = model.predict_proba(X_current)
    
    # Add predictions to dataframe
    current_df['win_probability'] = predictions[:, 1]
    
    # For each position, rank candidates by win probability
    current_df['predicted_rank'] = current_df.groupby(['election_id', 'position_id'])['win_probability'] \
        .rank(ascending=False, method='dense')
    
    # Identify predicted winners (top rank for each position)
    current_df['is_predicted_winner'] = (current_df['predicted_rank'] == 1).astype(int)
    
    # Save results
    output_path = current_election_data_path.replace('.csv', '_predictions.csv')
    current_df.to_csv(output_path, index=False)
    
    print(f"Predictions saved to {output_path}")
    return current_df

# Example usage
current_predictions = predict_current_election('current_election_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'current_election_data.csv'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_predictions(predictions_df):
    # Get predicted winners
    winners = predictions_df[predictions_df['is_predicted_winner'] == 1]
    
    # 1. Institution dominance analysis
    inst_win_counts = winners['candidate_institution'].value_counts()
    plt.figure(figsize=(12, 6))
    inst_win_counts.plot(kind='bar')
    plt.title('Predicted Wins by Institution')
    plt.ylabel('Number of Positions')
    plt.tight_layout()
    plt.savefig('institution_wins.png')
    plt.close()
    
    # 2. Position competitiveness analysis
    position_competitiveness = predictions_df.groupby(['position_id', 'position_name'])['win_probability'] \
        .agg(['max', 'min', 'mean'])
    position_competitiveness['competitiveness'] = 1 - (
        position_competitiveness['max'] - position_competitiveness['mean']
    )
    
    # 3. Save analysis report
    analysis_report = f"""
    ELECTION PREDICTION ANALYSIS REPORT
    ==================================
    
    Total Positions: {len(predictions_df['position_id'].unique())}
    Total Candidates: {len(predictions_df)}
    
    Institution Dominance:
    {inst_win_counts.to_string()}
    
    Most Competitive Positions (closest races):
    {position_competitiveness.sort_values('competitiveness', ascending=False).head(10).to_string()}
    
    Least Competitive Positions:
    {position_competitiveness.sort_values('competitiveness').head(10).to_string()}
    """
    
    with open('prediction_analysis_report.txt', 'w') as f:
        f.write(analysis_report)
    
    return {
        'institution_wins': inst_win_counts,
        'position_competitiveness': position_competitiveness
    }

# Example usage
analysis_results = analyze_predictions(current_predictions)